# 3D interpolation: filling in between 2D slices

Notebook implementation by Andy Thai

Sample implementation: https://github.com/fiji/VIB/blob/master/src/main/java/Implicit_Interpolate_Binary.java

Run from the `notebooks/` folder after `pip install -e .`. Paths are relative to it: datasets live in `../data/`, results go to `../output/`.

Requires `../data/npy/lv.npy`: binary left-ventricle masks shaped `(T, Z, H, W)`.

Set up our interpolation function f(x):

$
f(x) = \sum_{j=1}^k d_j \phi(x - c_j) + P(x) 
$

<br>

We want to solve for $d_j$ and the coefficients of $P(x)$. To solve for $d_j$, we can substitute the right side of the above equation for $f(\textbf{c}_i)$ for the constraints $h_i = f(\textbf{c}_i)$ to get the below equation, where $h_i$ is the height value:

$
h_i = \sum_{j=1}^k d_j \phi(c_i - c_j) + P(c_i) 
$

<br>
We can represent this equation as the matrix in the 3D interpolation case, where $h_i$ is corresponding x or y coordinate, depending on whether you are solving for $f'_{x}$ or $f'_{y}$:

$
\begin{bmatrix}
\phi_{11} & \phi_{12} & \ldots & \phi_{1k} & 1 & c_1^x & c_1^y & c_1^z \\
\phi_{21} & \phi_{22} & \ldots & \phi_{2k} & 1 & c_2^x & c_2^y & c_2^z \\
\vdots & \vdots & \ldots & \vdots & \vdots & \vdots & \vdots & \vdots \\
\phi_{k1} & \phi_{k2} & \ldots & \phi_{kk} & 1 & c_k^x & c_k^y & c_k^z \\
1 & 1 & \ldots & 1 & 0 & 0 & 0 & 0 \\
c_1^x & c_2^x & \ldots & c_k^x & 0 & 0 & 0 & 0 \\
c_1^y & c_2^y & \ldots & c_k^y & 0 & 0 & 0 & 0 \\
c_1^z & c_2^z & \ldots & c_k^z & 0 & 0 & 0 & 0 \\
\end{bmatrix}
\begin{bmatrix}
d_1 \\ 
d_2 \\
\vdots \\
d_k \\
p_0 \\
p_1 \\
p_2 \\
p_3 \\
\end{bmatrix}
=
\begin{bmatrix}
h_1 \\ 
h_2 \\
\vdots \\
h_k \\
0 \\
0 \\
0 \\
0 \\
\end{bmatrix}
$

<br>
Let $\phi_{ij} = \phi(\textbf{c}_i - \textbf{c}_j)$

$p_0, p_1, p_2, p_3$ can also be written as $p_0, p_x, p_y, p_z$ respectively.



## Import libraries

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from implicit_heart import ImplicitFunction
from implicit_heart import metrics

## Load a stack of 2D slices

One time-step of the LV volume; each slice becomes a constraint plane at its own `t`.

In [ ]:
volume = np.load("../data/npy/lv.npy")
volume[volume != 0] = 1

M = volume[15, 2:]  # (slices, H, W)
print(M.shape)

for i in range(M.shape[0]):
    plt.title(str(i)); plt.imshow(M[i]); plt.show()

## Fit the implicit function and interpolate between slices

`tmax=7` places the 8 slices at t = 0, 1, ..., 7, so fractional `t` falls between slices.

In [ ]:
start = time.time()
imp = ImplicitFunction(M, tmax=7)
print(f"Fit time: {time.time() - start:.1f}s")

img = imp.interpolate(0)
print("t=0 with boundary (0), inside (+1) and outside (-1) constraints")
for target in ([0], [1], [-1]):
    imp.data.show(0, img, target)

for t in [0.25, 0.5, 0.75, 1, 5, 7]:
    img = imp.interpolate(t)
    plt.title(f"t = {t}"); plt.imshow(img); plt.show()

## Two distant slices

Interpolate between only the first and last slice.

In [ ]:
M2 = np.array([M[0], M[7]])
imp = ImplicitFunction(M2, tmax=1)

for t in [0, 0.25, 0.5, 0.75, 1]:
    img = imp.interpolate(t)
    plt.title(f"t = {t}"); plt.imshow(img); plt.axis(False); plt.show()